# Compare Performance of all Finetuned Models

## **DistillBERT**

- run_0
- run_1
- run_2

## **BioBERT**

- run_0

In [ ]:
import sys, os, json

PARENT_DIR = os.path.abspath(os.path.join(os.getcwd(), ".."))
if PARENT_DIR not in sys.path:
    sys.path.insert(0, PARENT_DIR)

from gcp_utils import download_from_gcs, list_bucket_files
from config import settings

BUCKET_NAME = settings.BUCKET_NAME  # "ner_training_data_results"
VERSION = settings.VERSION

# Load the labels (same for distill and biobert)
with open("data/distillbert_splits/test.jsonl", "r") as f:
    test_data = []
    for line in f:
        test_data.append(line)

with open("data/id2label.json", "r") as f:
    id2label = json.load(f)
with open("data/label2id.json", "r") as f:
    label2id = json.load(f)
# Convert id2label keys from strings to integers (JSON loads keys as strings)
if any(isinstance(k, str) for k in id2label.keys()):
    id2label = {int(k): v for k, v in id2label.items()}


# Helper to print summary
def run_summary(run_summary):
    # Unpack model information
    model_name = run_summary.get('model_name')
    training_time = run_summary.get('training_time_minutes')
    hyper = run_summary.get('hyperparameters', {})
    epoch = hyper.get('epoch')
    lr = hyper.get('lr')
    bs = hyper.get('batch_size')
    warmup_ratio = hyper.get('warmup_ratio')
    pth = hyper.get('push_to_hub')

    print("-"*20)
    print(f"\t\t🤖 Model: {model_name}")
    print("-"*20)
    print(f"⏱️ Training time: {training_time}")
    print("🛠️ Hyperparameters:")
    print(f"\t🚀 Epoch: {epoch}")
    print(f"\t🚀 LR: {lr}")
    print(f"\t🚀 Batch Size: {bs}")
    print(f"\t🚀 Warmup Ratio: {warmup_ratio}")
    print(f"\t⬆️ Push to hub: {pth}")
    return

def display_best_metrics(results_dict, model_name):
    import pandas as pd

    runs = []
    for run_name, metrics_dict in results_dict.items():
        row = {
            "run": run_name,
            "val_f1": metrics_dict["validation_metrics"]["f1"],
            "val_precision": metrics_dict["validation_metrics"]["precision"],
            "val_recall": metrics_dict["validation_metrics"]["recall"],
            "test_f1": metrics_dict["test_metrics"]["f1"],
            "test_precision": metrics_dict["test_metrics"]["precision"],
            "test_recall": metrics_dict["test_metrics"]["recall"],
        }
        runs.append(row)
    df = pd.DataFrame(runs).set_index("run")

    def get_max_info(column):
        idx_max = df[column].idxmax()
        val_max = df.loc[idx_max, column]
        return idx_max, val_max

    summary = []
    for col, label in [
        ("val_f1", "Validation F1"), 
        ("val_precision", "Validation Precision"), 
        ("val_recall", "Validation Recall"),
        ("test_f1", "Test F1"), 
        ("test_precision", "Test Precision"), 
        ("test_recall", "Test Recall")
    ]:
        run, val = get_max_info(col)
        summary.append((label, run, val))

    print(f"🏆 Best {model_name} runs by metric:\n")
    for label, run, val in summary:
        print(f"  • {label:<19s}: {run} ({val:.4f})")
    print("\nFull summary table:")
    display(df)

In [10]:
VERSION

'v01'

# DistillBERT Comparison

In [ ]:
# ==============================
# Load distillBERT models
# ==============================
MODEL_NAME = "distilbert-base-uncased" 
for idx in [0,1,2]:
    GCS_MODEL_PATH = f"{VERSION}/runs/{MODEL_NAME}/run_{idx}"
    LOCAL_MODEL_DIR = f"./downloaded_models/{MODEL_NAME}/run_{idx}"
    print(f"Downloading model from gs://{BUCKET_NAME}/{GCS_MODEL_PATH}...")
    downloaded_path = download_from_gcs(
        gcs_path=GCS_MODEL_PATH,
        local_path=LOCAL_MODEL_DIR,
        bucket_name=BUCKET_NAME
    )


✅  Downloaded all files from the folder
✓ Downloaded directory: gs://ner_training_data_results/v01/runs/distilbert-base-uncased/run_0 (24 files) → ./downloaded_models/distilbert-base-uncased/run_0
✅  Downloaded all files from the folder
✓ Downloaded directory: gs://ner_training_data_results/v01/runs/distilbert-base-uncased/run_1 (24 files) → ./downloaded_models/distilbert-base-uncased/run_1
✅  Downloaded all files from the folder
✓ Downloaded directory: gs://ner_training_data_results/v01/runs/distilbert-base-uncased/run_2 (24 files) → ./downloaded_models/distilbert-base-uncased/run_2


In [17]:
distill_val_test = {}
for idx in range(3):
    path = f"downloaded_models/distilbert-base-uncased/run_{idx}/summary.json"
    with open(path, "r") as f:
        summary = json.load(f)
    print(f"IDX: {idx}")
    run_summary(summary)
    distill_val_test[f"run_{idx}"] = {
        "validation_metrics" : summary["validation_metrics"],
        "test_metrics" : summary["test_metrics"]
    }

IDX: 0
--------------------
		🤖 Model: distilbert-base-uncased
--------------------
⏱️ Training time: 3.23
🛠️ Hyperparameters:
	🚀 Epoch: 20
	🚀 LR: 5e-05
	🚀 Batch Size: 32
	🚀 Warmup Ratio: 0.1
	⬆️ Push to hub: True
IDX: 1
--------------------
		🤖 Model: distilbert-base-uncased
--------------------
⏱️ Training time: 2.61
🛠️ Hyperparameters:
	🚀 Epoch: 15
	🚀 LR: 3e-05
	🚀 Batch Size: 32
	🚀 Warmup Ratio: 0.1
	⬆️ Push to hub: True
IDX: 2
--------------------
		🤖 Model: distilbert-base-uncased
--------------------
⏱️ Training time: 2.09
🛠️ Hyperparameters:
	🚀 Epoch: 10
	🚀 LR: 2e-05
	🚀 Batch Size: 16
	🚀 Warmup Ratio: 0.1
	⬆️ Push to hub: True


In [ ]:
display_best_metrics(distill_val_test, model_name = "DistilBERT")

🏆 Best DistilBERT runs by metric:

  • Validation F1      : run_0 (0.5965)
  • Validation Precision: run_0 (0.5081)
  • Validation Recall  : run_0 (0.7223)
  • Test F1            : run_0 (0.6241)
  • Test Precision     : run_0 (0.5377)
  • Test Recall        : run_0 (0.7436)

Full summary table:


,val_f1,val_precision,val_recall,test_f1,test_precision,test_recall
run,,,,,,
run_0,0.596532,0.508074,0.722284,0.624060,0.537652,0.743561
run_1,0.509751,0.425047,0.636618,0.519305,0.435112,0.643897
run_2,0.456891,0.376864,0.580067,0.463620,0.383909,0.585106


*Best DistillBERT Model: run_0*

# BioBERT Comparison

In [ ]:
# ==============================
# Load BioBERT models
# ==============================
MODEL_NAME = "dmis-lab/biobert-base-cased-v1.1"
for idx in [2]:
    GCS_MODEL_PATH = f"{VERSION}/runs/{MODEL_NAME}/run_{idx}"
    LOCAL_MODEL_DIR = f"./downloaded_models/{MODEL_NAME}/run_{idx}"
    print(f"Downloading model from gs://{BUCKET_NAME}/{GCS_MODEL_PATH}...")
    downloaded_path = download_from_gcs(
        gcs_path=GCS_MODEL_PATH,
        local_path=LOCAL_MODEL_DIR,
        bucket_name=BUCKET_NAME
    )

✅  Downloaded all files from the folder
✓ Downloaded directory: gs://ner_training_data_results/v01/runs/dmis-lab/biobert-base-cased-v1.1/run_2 (24 files) → ./downloaded_models/dmis-lab/biobert-base-cased-v1.1/run_2


In [20]:
biobert_val_test = {}
for idx in range(3):
    path = f"downloaded_models/{MODEL_NAME}/run_{idx}/summary.json"
    if not os.path.exists(path):
        continue
    with open(path, "r") as f:
        summary = json.load(f)
    print(f"IDX: {idx}")
    run_summary(summary)
    biobert_val_test[f"run_{idx}"] = {
        "validation_metrics" : summary["validation_metrics"],
        "test_metrics" : summary["test_metrics"]
    }

IDX: 2
--------------------
		🤖 Model: dmis-lab/biobert-base-cased-v1.1
--------------------
⏱️ Training time: 7.01
🛠️ Hyperparameters:
	🚀 Epoch: 20
	🚀 LR: 5e-05
	🚀 Batch Size: 16
	🚀 Warmup Ratio: 0.1
	⬆️ Push to hub: True


In [21]:
display_best_metrics(distill_val_test, model_name = "DistilBERT")

🏆 Best DistilBERT runs by metric:

  • Validation F1      : run_2 (0.8090)
  • Validation Precision: run_2 (0.7445)
  • Validation Recall  : run_2 (0.8858)
  • Test F1            : run_2 (0.7907)
  • Test Precision     : run_2 (0.7222)
  • Test Recall        : run_2 (0.8735)

Full summary table:


,val_f1,val_precision,val_recall,test_f1,test_precision,test_recall
run,,,,,,
run_0,0.596532,0.508074,0.722284,0.624060,0.537652,0.743561
run_1,0.509751,0.425047,0.636618,0.519305,0.435112,0.643897
run_2,0.809000,0.744471,0.885778,0.790674,0.722222,0.873460
